# Pipeline Tiền Xử Lý
| Mục | Chi tiết |
|---|---|
| **Mô hình** | RoBERTa-base → 8-label binary classification |
| **Aspects** | Scent / Longevity / Packaging / Service (× positive / negative) |
| **Input** | `data.json` (raw scrape từ yslbeauty.com) |
| **Output** | `df_final.csv` + `pos_weight.json` |

## 0. Import thư viện

In [1]:
import json
import re

!pip install emoji
!pip install -q transformers
import emoji
import numpy as np
import pandas as pd
from transformers import RobertaTokenizerFast

TOKENIZER = RobertaTokenizerFast.from_pretrained("roberta-base")
print("✅ Tokenizer loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 4.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded.


## 1. Load dữ liệu

In [2]:
# Cài đặt gdown (nếu chưa có)
!pip install -q gdown

# Tải file JSON từ link Google Drive
!gdown https://drive.google.com/uc?id=17jgUe6k8nZBedN867q2ef3H8Y1K0khrD -O data.json

print("✅ Tải file thành công!")

Downloading...
From: https://drive.google.com/uc?id=17jgUe6k8nZBedN867q2ef3H8Y1K0khrD
To: /content/data.json
100% 25.6M/25.6M [00:00<00:00, 45.0MB/s]
✅ Tải file thành công!


In [3]:
with open("data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Raw: {len(df):,} reviews, {df.shape[1]} cột")
df.head()

Raw: 50,239 reviews, 5 cột


,id,title,text,star,labels
0,bv-review-text-248964196,Lost,Still have not received it. It is lost obvious...,1,"{'longevity': 'none', 'packaging': 'none', 'sc..."
1,bv-review-text-248930739,Very Poor Product Delivery,The product NEVER reached me even after 30 day...,1,"{'longevity': 'none', 'packaging': 'none', 'sc..."
2,bv-review-text-298544599,Not worth it,Scent doesn t stay long,1,"{'longevity': 'negative', 'packaging': 'none',..."
3,bv-review-text-247286197,THE SPRAYING MECHANISM HAS FALLEN INSIDE THE BOTT,I ATTEMPTED TO USE THE PERFUME last week AND D...,1,"{'longevity': 'none', 'packaging': 'negative',..."
4,bv-review-text-246920559,Yuck,"Too strong, very masculine and serious, gave m...",1,"{'longevity': 'none', 'packaging': 'none', 'sc..."


## 2. Lọc độ dài (token-based)

Lọc theo **số token** thực tế của RoBERTa tokenizer, không phải số ký tự.

- **< 10 tokens**: Quá ngắn, không đủ thông tin nhận diện aspect
- **> 512 tokens**: Vượt giới hạn cứng của RoBERTa (max_position_embeddings = 512)

> ⚠️ Lý do không dùng character count: 512 ký tự ≈ 100–140 token, lọc theo ký tự vừa bỏ review hợp lệ dài (upper bound quá chặt) vừa không ngăn được truncation thật (lower bound không đủ ý nghĩa).

In [4]:
def count_tokens(text: str) -> int:
    """Đếm số token RoBERTa (bao gồm [CLS] và [SEP])."""
    return len(TOKENIZER.encode(text, truncation=False, add_special_tokens=True))


df["token_length"] = df["text"].apply(count_tokens)
df = df[(df["token_length"] >= 10) & (df["token_length"] <= 512)].reset_index(drop=True)
print(f"Sau lọc độ dài   : {len(df):,}")
print(f"  → token P50={df['token_length'].quantile(0.5):.0f}  "
      f"P90={df['token_length'].quantile(0.9):.0f}  "
      f"P99={df['token_length'].quantile(0.99):.0f}")

Token indices sequence length is longer than the specified maximum sequence length for this model (579 > 512). Running this sequence through the model will result in indexing errors


Sau lọc độ dài   : 47,846
  → token P50=135  P90=313  P99=476


## 3. Xóa duplicate

In [5]:
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
print(f"Sau xóa duplicate: {len(df):,}")

Sau xóa duplicate: 47,768


## 4. Làm sạch text

In [6]:
def clean_text(text: str) -> str:
    """Chuẩn hoá text: lowercase, bỏ emoji, giữ ký tự Latin + dấu câu."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = emoji.replace_emoji(text, replace="")          # bỏ emoji
    text = re.sub(r"[^a-z0-9\s.,!?;:\-\'\"]", " ", text)   # giữ Latin + dấu câu
    text = re.sub(r"\s+([.,!?;])", r"\1", text)           # bỏ space trước dấu câu
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["text_clean"]  = df["text"].apply(clean_text)
df["title_clean"] = df["title"].fillna("").apply(clean_text)

# full_text = title + text  →  input duy nhất vào RoBERTa tokenizer
df["full_text"] = (df["title_clean"] + ". " + df["text_clean"]).str.strip(". ")

print("Ví dụ full_text:")
df[["title_clean", "text_clean", "full_text"]].head(3)

Ví dụ full_text:


,title_clean,text_clean,full_text
0,lost,still have not received it. it is lost obvious...,lost. still have not received it. it is lost o...
1,very poor product delivery,the product never reached me even after 30 day...,very poor product delivery. the product never ...
2,not worth it,scent doesn t stay long,not worth it. scent doesn t stay long


## 5. Parse nhãn từ cột `labels`

`labels` là dict dạng `{'scent': 'positive', 'longevity': 'none', ...}`

Mỗi aspect sinh ra **2 cột binary**: `{aspect}_positive` và `{aspect}_negative`

In [7]:
ASPECTS   = ["scent", "longevity", "packaging", "service"]   # bỏ sillage
LABEL_COLS = [f"{a}_{p}" for a in ASPECTS for p in ("positive", "negative")]

print("Label columns:", LABEL_COLS)


def parse_labels(label_dict) -> pd.Series:
    if isinstance(label_dict, str):
        label_dict = json.loads(label_dict.replace("'", '"'))
    row = {}
    for aspect in ASPECTS:
        val = label_dict.get(aspect, "none")
        row[f"{aspect}_positive"] = int(val == "positive")
        row[f"{aspect}_negative"] = int(val == "negative")
    return pd.Series(row)


df[LABEL_COLS] = df["labels"].apply(parse_labels)
df[LABEL_COLS].head()

Label columns: ['scent_positive', 'scent_negative', 'longevity_positive', 'longevity_negative', 'packaging_positive', 'packaging_negative', 'service_positive', 'service_negative']


,scent_positive,scent_negative,longevity_positive,longevity_negative,packaging_positive,packaging_negative,service_positive,service_negative
0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,1
2,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,1
4,0,1,0,0,0,0,0,0


## 6. Bỏ review không có nhãn (`none`)

In [8]:
df["num_labels"] = df[LABEL_COLS].sum(axis=1)
df = df[df["num_labels"] > 0].reset_index(drop=True)
print(f"Sau bỏ none      : {len(df):,}")

Sau bỏ none      : 46,569


## 7. Bỏ review có nhãn mâu thuẫn

`positive=1` và `negative=1` cùng lúc cho 1 aspect → lỗi annotation

In [9]:
conflict_mask = pd.Series(False, index=df.index)
for aspect in ASPECTS:
    conflict_mask |= (
        (df[f"{aspect}_positive"] == 1) & (df[f"{aspect}_negative"] == 1)
    )

n_conflict = conflict_mask.sum()
df = df[~conflict_mask].reset_index(drop=True)
print(f"Sau bỏ conflict  : {len(df):,}  (loại {n_conflict:,} mâu thuẫn)")

Sau bỏ conflict  : 46,569  (loại 0 mâu thuẫn)


## 8. Tính `pos_weight` cho Loss Function

$$\\text{pos\_weight}_i = \\max\\!\\left(1,\\; \\frac{N_{neg}}{N_{pos}}\\right)$$

Clipping về $\\geq 1$ để tránh **down-weight** nhãn majority-positive (ví dụ `scent_positive` chiếm ~91% → $N_{neg}/N_{pos} < 1$ → không clip sẽ phạt nhầm chiều).

Lưu ra `pos_weight.json` để truyền vào `BCEWithLogitsLoss(pos_weight=...)` trong training notebook.

In [10]:
total = len(df)
pos_weight = {}

rows = []
for col in LABEL_COLS:
    pos = int(df[col].sum())
    neg = total - pos
    raw_w = neg / pos if pos > 0 else 999.0
    w     = round(max(1.0, raw_w), 4)   # clamp ≥ 1: không down-weight class majority
    pos_weight[col] = w
    rows.append({"Nhãn": col, "Pos": pos, "%": round(pos / total * 100, 1),
                 "raw_w": round(raw_w, 4), "pos_weight (clipped)": w})

pd.DataFrame(rows).set_index("Nhãn")

,Pos,%,raw_w,pos_weight (clipped)
Nhãn,,,,
scent_positive,42490,91.2,0.0960,1.0000
scent_negative,2317,5.0,19.0988,19.0988
longevity_positive,17645,37.9,1.6392,1.6392
longevity_negative,2078,4.5,21.4105,21.4105
packaging_positive,11443,24.6,3.0696,3.0696
packaging_negative,909,2.0,50.2310,50.2310
service_positive,1663,3.6,27.0030,27.0030
service_negative,934,2.0,48.8597,48.8597


## 9. Lưu output

In [11]:
df_final = df[["id", "full_text"] + LABEL_COLS].copy()

df_final.to_csv("df_final.csv", index=False)

with open("pos_weight.json", "w") as f:
    json.dump(pos_weight, f, indent=2)

print(f"df_final : {df_final.shape}  → df_final.csv")
print(f"Cột      : id | full_text | {'| '.join(LABEL_COLS)}")
print("pos_weight.json  → truyền vào BCEWithLogitsLoss(pos_weight=...) lúc train")
print("Lưu ý: các giá trị đã được clamp max(1, N_neg/N_pos)")

df_final.head()

df_final : (46569, 10)  → df_final.csv
Cột      : id | full_text | scent_positive| scent_negative| longevity_positive| longevity_negative| packaging_positive| packaging_negative| service_positive| service_negative
pos_weight.json  → truyền vào BCEWithLogitsLoss(pos_weight=...) lúc train
Lưu ý: các giá trị đã được clamp max(1, N_neg/N_pos)


,id,full_text,scent_positive,scent_negative,longevity_positive,longevity_negative,packaging_positive,packaging_negative,service_positive,service_negative
0,bv-review-text-248964196,lost. still have not received it. it is lost o...,0,0,0,0,0,0,0,1
1,bv-review-text-248930739,very poor product delivery. the product never ...,0,0,0,0,0,0,0,1
2,bv-review-text-298544599,not worth it. scent doesn t stay long,0,0,0,1,0,0,0,0
3,bv-review-text-247286197,the spraying mechanism has fallen inside the b...,0,0,0,0,0,1,0,1
4,bv-review-text-246920559,"yuck. too strong, very masculine and serious, ...",0,1,0,0,0,0,0,0
